In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import pyulog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R_
from pid_optimizer.plant_model import PlantModel
from pid_optimizer.gains import HERMIT_REFERENCE_GAINS
from pid_optimizer.pid_simulator import PX4Simulator, compute_rmse

LOG_PATH   = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
PLANT_PATH = "../artifacts/plant_model.json"

plant = PlantModel.load(PLANT_PATH)
log   = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0    = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

def interp_col(df, col, t_out):
    mask = ~df[col].isna()
    if mask.sum() < 2:
        return np.zeros(len(t_out))
    return np.interp(t_out, df["t_s"][mask].values, df[col][mask].values)

DT    = 0.004
T_MAX = 30.0

# Load all topics needed
pos_sp_df = to_df(log, "vehicle_local_position_setpoint")
att_sp_df = to_df(log, "vehicle_attitude_setpoint")
pos_df    = to_df(log, "vehicle_local_position")
att_df    = to_df(log, "vehicle_attitude")
torque_df = to_df(log, "vehicle_torque_setpoint")   # xyz[0..2] in N*m
thrust_df = to_df(log, "vehicle_thrust_setpoint")   # xyz[2] = normalized throttle
rates_df  = to_df(log, "vehicle_angular_velocity")

print(f"Plant: {plant}")
print(f"Log topics loaded. torque msgs={len(torque_df)}, thrust msgs={len(thrust_df)}")


Plant: PlantModel(mass_kg=1.8, kT=23.807374954223633, tau_motor_s=0.005003370182525293, inertia=Inertia(Ixx=0.01728191388581314, Iyy=0.009356793261798222, Izz=0.021638707147611364), drag=Drag(kD_xy=0.15, kD_z=0.2), arm_length_m=0.17, source_log='../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg', fit_rmse={})
Log topics loaded. torque msgs=68491, thrust msgs=68523


In [2]:

# Build simulation time axis aligned to the available high-rate data
t_start_raw = max(torque_df["t_s"].min(), pos_df["t_s"].min(), att_df["t_s"].min())
t_end       = min(torque_df["t_s"].max(), att_df["t_s"].max(), T_MAX)

# Convert normalized thrust to physical force [N].
# vehicle_thrust_setpoint.xyz[2] = mean per-motor throttle (0 to -1, negative = upward).
# F_total = kT_per_motor * n_motors * throttle^2  (kT identified from hover sysid)
def _thr_to_F(thr_arr, kT=plant.kT):
    return kT * 4.0 * thr_arr**2   # N, always positive

# Calibrate thrust scale from hover phase.
# During stable hover, F_total = mass * g.  Find the segment where the drone is at
# a stable altitude (|vz| < 0.1 m/s) and z well above ground (z < -0.5m).
_t_cal   = np.arange(t_start_raw, t_end, DT)
_z_cal   = np.interp(_t_cal, pos_df["t_s"].values, pos_df["z"].values)
_vz_cal  = np.interp(_t_cal, pos_df["t_s"].values,
                     pos_df["vz"].values if "vz" in pos_df.columns else np.gradient(pos_df["z"].values, pos_df["t_s"].values))
_thr_cal = np.abs(np.interp(_t_cal, thrust_df["t_s"].values, thrust_df["xyz[2]"].values))
hover_mask = (_z_cal < -0.5) & (np.abs(_vz_cal) < 0.1) & (_thr_cal > 0.1)

if hover_mask.sum() > 50:
    thr_hover  = np.mean(_thr_cal[hover_mask])
    F_hover_needed = plant.mass_kg * 9.81
    kT_calibrated  = F_hover_needed / (4.0 * thr_hover**2)
    print(f"Hover calibration: mean throttle={thr_hover:.4f}, "
          f"kT_calibrated={kT_calibrated:.2f} N/thr^2  (plant.kT={plant.kT:.2f})")
else:
    kT_calibrated = plant.kT
    print(f"Could not find hover segment (n={hover_mask.sum()}), using plant.kT={plant.kT:.2f}")

# Full simulation time axis
t_sim   = np.arange(t_start_raw, t_end, DT)
thr_raw = np.abs(np.interp(t_sim, thrust_df["t_s"].values, thrust_df["xyz[2]"].values))
F_total = _thr_to_F(thr_raw, kT=kT_calibrated)
thrust_sim = pd.DataFrame({"F": F_total})

hover_F = plant.mass_kg * 9.81
print(f"Sim time: [{t_sim[0]:.2f}, {t_sim[-1]:.2f}] s  ({len(t_sim)} steps)")
print(f"Mean simulated thrust: {F_total.mean():.2f} N  (hover weight = {hover_F:.2f} N)")
print(f"Thrust range: [{F_total.min():.2f}, {F_total.max():.2f}] N")

# Interpolate real attitude (Euler XYZ) from vehicle_attitude quaternion onto t_sim.
q_arr     = att_df[["q[0]","q[1]","q[2]","q[3]"]].values      # w,x,y,z
euler_arr = R_.from_quat(q_arr[:, [1,2,3,0]]).as_euler("xyz")  # roll, pitch, yaw
t_att     = att_df["t_s"].values

euler_sim = pd.DataFrame({
    "roll":  np.interp(t_sim, t_att, euler_arr[:, 0]),
    "pitch": np.interp(t_sim, t_att, euler_arr[:, 1]),
    "yaw":   np.interp(t_sim, t_att, euler_arr[:, 2]),
})

# Initial state from ULG at t_start
def _idx_at(df, t):
    return min(np.searchsorted(df["t_s"].values, t), len(df) - 1)

x0_pos  = pos_df[["x","y","z"]].iloc[_idx_at(pos_df, t_sim[0])].values
vel_cols = [c for c in ["vx","vy","vz"] if c in pos_df.columns]
x0_vel  = pos_df[vel_cols].iloc[_idx_at(pos_df, t_sim[0])].values if vel_cols else np.zeros(3)
x0 = np.concatenate([x0_pos, x0_vel])
print(f"Initial state: pos={x0_pos.round(3)}, vel={x0_vel.round(3)}")


Hover calibration: mean throttle=0.4440, kT_calibrated=22.39 N/thr^2  (plant.kT=23.81)
Sim time: [0.06, 30.00] s  (7485 steps)
Mean simulated thrust: 14.71 N  (hover weight = 17.66 N)
Thrust range: [0.00, 20.08] N
Initial state: pos=[ 0.018  0.038 -0.055], vel=[0.001 0.    0.009]


In [3]:

# Translational-only simulation:
#   - attitude taken from real flight log (not integrated)
#   - only position + velocity propagated forward
#   - avoids torque-bias accumulation that causes 6DOF open-loop divergence
sim    = PX4Simulator(plant, HERMIT_REFERENCE_GAINS)
result = sim.run_translational(thrust_sim, euler_sim, dt=DT, x0=x0)
print(f"Simulation done. result shape: {result.shape}")
print(f"Position range: x=[{result['x'].min():.3f}, {result['x'].max():.3f}]  "
      f"y=[{result['y'].min():.3f}, {result['y'].max():.3f}]  "
      f"z=[{result['z'].min():.3f}, {result['z'].max():.3f}]")

# Precompute time arrays and masks for plotting
t_plot      = t_sim[:len(result)]
t_att       = att_df["t_s"].values
t_att_sp    = att_sp_df["t_s"].values
mask_pos    = pos_df["t_s"].values   <= T_MAX
mask_att    = t_att   <= T_MAX
mask_att_sp = t_att_sp <= T_MAX

# Attitude setpoints -> Euler for setpoint curve
q_sp     = att_sp_df[["q_d[0]","q_d[1]","q_d[2]","q_d[3]"]].values
euler_sp = R_.from_quat(q_sp[:, [1,2,3,0]]).as_euler("xyz")

# Figure 1: Position (3 curves) + Attitude (setpoint & real)
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for i, (col, label) in enumerate([("x","X (m)"), ("y","Y (m)"), ("z","Z (m)")]):
    ax = axes[i, 0]
    ax.plot(t_plot, result[col].values,
            color="tab:blue", label="simulated", linewidth=1.5)
    ax.plot(pos_df["t_s"].values[mask_pos], pos_df[col].values[mask_pos],
            color="tab:orange", label="real (EKF)", alpha=0.8, linewidth=1)
    ax.plot(t_plot, interp_col(pos_sp_df, col, t_plot),
            color="black", label="setpoint", linestyle="--", alpha=0.5)
    ax.set_ylabel(label); ax.legend(fontsize=8)
    ax.set_title(f"Position {label}"); ax.grid(True, alpha=0.3)

for i, (idx, label) in enumerate([(0,"Roll (deg)"), (1,"Pitch (deg)"), (2,"Yaw (deg)")]):
    ax = axes[i, 1]
    ax.plot(t_att[mask_att], np.rad2deg(euler_arr[mask_att, idx]),
            color="tab:orange", label="real (EKF)", alpha=0.9, linewidth=1)
    ax.plot(t_att_sp[mask_att_sp], np.rad2deg(euler_sp[mask_att_sp, idx]),
            color="black", label="setpoint", linestyle="--", alpha=0.5)
    ax.set_ylabel(label); ax.legend(fontsize=8)
    ax.set_title(f"Attitude {label}"); ax.grid(True, alpha=0.3)
    ax.annotate("attitude taken from log\n(not simulated)",
                xy=(0.98, 0.05), xycoords="axes fraction",
                fontsize=7, ha="right", color="gray")

for ax in axes[-1, :]:
    ax.set_xlabel("Time (s)")
plt.suptitle("Translational sim (attitude from log): setpoint vs real vs simulated", fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/sim_pos_att.png", dpi=120); plt.close()
print("Saved /tmp/sim_pos_att.png")


Simulation done. result shape: (7485, 6)
Position range: x=[-2.056, 5.354]  y=[-17.558, 0.224]  z=[-3.278, -0.005]
Saved /tmp/sim_pos_att.png


In [4]:

# RMSE vs real EKF position
actual_pos = pd.DataFrame({
    "x": interp_col(pos_df, "x", t_sim),
    "y": interp_col(pos_df, "y", t_sim),
    "z": interp_col(pos_df, "z", t_sim),
})
rmse = compute_rmse(result[["x","y","z"]], actual_pos[["x","y","z"]], cols=["x","y","z"])
print("Position RMSE vs real (EKF)  [translational sim, attitude from log]:")
for col, val in rmse.items():
    sig_range = actual_pos[col].max() - actual_pos[col].min()
    pct = 100 * val / sig_range if sig_range > 0.05 else float("nan")
    status = "OK" if pct < 20 else "WARN"
    print(f"  {col}: {val:.3f} m  ({pct:.1f}% of signal range)  [{status}]")

print()
print("Remaining RMSE is due to thrust model (kT, hover throttle) and drag.")
print("GA in notebook 04 will optimize [mass, kT, kD_xy, kD_z] to minimize this.")


Position RMSE vs real (EKF)  [translational sim, attitude from log]:
  x: 2.276 m  (111.1% of signal range)  [WARN]
  y: 7.134 m  (929.5% of signal range)  [WARN]
  z: 0.905 m  (74.1% of signal range)  [WARN]

Remaining RMSE is due to thrust model (kT, hover throttle) and drag.
GA in notebook 04 will optimize [mass, kT, kD_xy, kD_z] to minimize this.
